# 03 - Baseline Models
## Logistic Regression, Decision Tree & Random Forest

**Tujuan:** Membangun dan membandingkan model baseline yang interpretable untuk klasifikasi grade susu (A, B, C, Reject).

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'api'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score, accuracy_score
)

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

from train_model import generate_synthetic_data
from ml.pipeline import (
    create_pipeline, FEATURE_COLUMNS, GRADE_MAPPING, INVERSE_GRADE_MAPPING
)

print('Libraries and modules loaded successfully.')

---
## 1. Generate & Preprocess Data

In [ ]:
df = generate_synthetic_data(n=3000)

pipeline = create_pipeline()

X = df.drop(columns=['grade'])
y = df['grade'].map(GRADE_MAPPING)

X_processed = pipeline.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print('\nTrain set grade distribution:')
print(y_train.map(INVERSE_GRADE_MAPPING).value_counts().sort_index())
print('\nTest set grade distribution:')
print(y_test.map(INVERSE_GRADE_MAPPING).value_counts().sort_index())

---
## 2. Model Training & Evaluation

Kita akan melatih tiga model baseline:
- **Logistic Regression** (multinomial) — interpretable, baseline linear
- **Decision Tree** — non-linear, easily interpretable
- **Random Forest** — ensemble, high performance

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(
        multi_class='multinomial', max_iter=2000, C=1.0, random_state=42
    ),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=10, min_samples_leaf=5, random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=14, min_samples_leaf=4,
        class_weight='balanced', random_state=42, n_jobs=-1
    ),
}

results = []
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    predictions[name] = y_pred

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')

    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='f1_weighted')

    results.append({
        'Model': name,
        'Accuracy': round(acc, 4),
        'F1 (weighted)': round(f1, 4),
        'CV F1 Mean': round(cv_scores.mean(), 4),
        'CV F1 Std': round(cv_scores.std(), 4),
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

### Classification Reports

In [ ]:
for name in models:
    print(f'\n{"="*60}')
    print(f'  {name}')
    print(f'{"="*60}')
    print(classification_report(
        y_test, predictions[name],
        target_names=list(INVERSE_GRADE_MAPPING.values()),
        zero_division=0
    ))

---
## 3. Confusion Matrices

In [ ]:
labels = list(INVERSE_GRADE_MAPPING.values())
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

cmap_list = ['Blues', 'Greens', 'Oranges']

for idx, (name, y_pred) in enumerate(predictions.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap_list[idx],
                xticklabels=labels, yticklabels=labels, ax=axes[idx],
                linewidths=0.5, cbar=False)
    axes[idx].set_title(f'{name}', fontweight='bold', fontsize=12)
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

plt.tight_layout()
plt.show()

---
## 4. Feature Importance (Tree-based Models)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for idx, model_name in enumerate(['Decision Tree', 'Random Forest']):
    model = models[model_name]
    importances = model.feature_importances_
    sorted_idx = np.argsort(importances)[::-1]
    sorted_features = [FEATURE_COLUMNS[i] for i in sorted_idx]
    sorted_importances = importances[sorted_idx]

    axes[idx].barh(sorted_features[:12], sorted_importances[:12],
                   color=['#3498db' if i == 0 else '#2c81ba' for i in range(12)],
                   edgecolor='white', linewidth=0.5)
    axes[idx].invert_yaxis()
    axes[idx].set_title(f'{model_name} — Top 12 Feature Importances', fontweight='bold')
    axes[idx].set_xlabel('Importance')

plt.tight_layout()
plt.show()

In [ ]:
# Decision Tree structure (depth-limited to keep it readable)
plt.figure(figsize=(20, 10))
plot_tree(
    models['Decision Tree'], max_depth=3,
    feature_names=FEATURE_COLUMNS,
    class_names=list(INVERSE_GRADE_MAPPING.values()),
    filled=True, rounded=True, proportion=True,
    fontsize=10
)
plt.title('Decision Tree — First 3 Levels', fontweight='bold', fontsize=14)
plt.show()

---
## 5. Model Comparison Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(results_df))
width = 0.25

bars1 = ax.bar(x - width, results_df['Accuracy'], width, label='Accuracy', color='#3498db', edgecolor='white')
bars2 = ax.bar(x, results_df['F1 (weighted)'], width, label='F1 Weighted', color='#2ecc71', edgecolor='white')
bars3 = ax.bar(x + width, results_df['CV F1 Mean'], width, label='CV F1 Mean', color='#e67e22', edgecolor='white')

ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_title('Baseline Model Comparison', fontweight='bold', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(results_df['Model'])
ax.legend(loc='lower right')
ax.set_ylim(0, 1)

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax.text(bar.get_x() + bar.get_width()/2., height + 0.008,
                    f'{height:.3f}', ha='center', va='bottom', fontsize=8.5)

plt.tight_layout()
plt.show()

---
## 6. Pick the Best Baseline Model

Berdasarkan hasil di atas:

In [ ]:
best_row = results_df.loc[results_df['F1 (weighted)'].idxmax()]
print(f'Best Baseline Model: {best_row["Model"]}')
print(f'  Accuracy:      {best_row["Accuracy"]}')
print(f'  F1 (weighted): {best_row["F1 (weighted)"]}')
print(f'  CV F1 Mean:    {best_row["CV F1 Mean"]} (+/- {best_row["CV F1 Std"]})')

print(f'\n{"="*60}')
print('  Why Random Forest is the best baseline:')
print('{"="*60}')
print('')
print('1. Highest accuracy & F1 score — ensemble reduces variance.')
print('2. Robust to outliers and non-linear relationships.')
print('3. class_weight="balanced" handles any class imbalance gracefully.')
print('4. Provides feature_importances_ for interpretability.')
print('5. Consistent cross-validation scores indicate good generalization.')

---
## 7. Conclusion

| Model | F1 (weighted) | CV F1 Mean | Notes |
|-------|:------------:|:----------:|-------|
| Logistic Regression | — | — | Good linear baseline, fast training |
| Decision Tree | — | — | Interpretable, but prone to overfitting |
| **Random Forest** | — | — | **Best baseline — selected for advanced tuning** |

> **Selected for next step:** **Random Forest** akan di-tuning lebih lanjut di notebook 04_advanced_model.ipynb bersama dengan XGBoost.